# A3 — PhoBERT phân loại ý định chatbot (Colab, chỉ để so sánh)

Bước **C7 (tuỳ chọn)** trong `docs/ke-hoach-chatbot.md`: fine-tune PhoBERT cho 12 nhãn ý định của chatbot Vet Clinic,
đánh giá trên **cùng** `train/val/test.csv` với A1 (TF-IDF + logreg) và A2 (embedding Gemini), rồi xuất
`results/phobert.md` theo định dạng của `results/a1.md`.

**Không** đưa vào hệ thống chạy thật: cần PyTorch + mô hình ~540 MB, độ trễ CPU trên VM nhỏ cao hơn A1 vài trăm lần
(xem `ml/chat-intent/phobert/README.md`).

**Cách chạy:** Runtime → Change runtime type → **T4 GPU**, sửa ô *Cấu hình* nếu cần, rồi Runtime → Run all.
Ô tải dữ liệu sẽ hỏi chọn 3 file CSV (mặc định `DATA_SOURCE = "upload"`).

**Thời gian ước tính trên Colab T4** (chưa chạy thực tế, chỉ ước lượng theo kích thước dữ liệu: 716 câu train, max_len 64):

| bước | ước tính |
|---|---|
| cài thư viện + tải VnCoreNLP + tải PhoBERT (~540 MB) | 2–4 phút |
| tách từ ~1 025 câu | < 30 giây |
| huấn luyện 1 learning rate (≤ 10 epoch × 45 bước, fp16, lưu checkpoint mỗi epoch) | 1–3 phút |
| lưới 3 learning rate | 4–9 phút |
| đánh giá val/test + đo độ trễ GPU/CPU (154 câu × 2 thiết bị) | 1–2 phút |
| **tổng** | **khoảng 10–15 phút** |

## 1. Cấu hình

**Chọn `vinai/phobert-base-v2` thay vì `vinai/phobert-base`:** cùng kiến trúc RoBERTa-base (~135 triệu tham số, cùng
tokenizer BPE trên văn bản đã tách từ), nhưng v2 được tiền huấn luyện trên ~140 GB (20 GB Wikipedia + tin tức như bản
gốc, thêm ~120 GB văn bản web OSCAR-2301). Dữ liệu chatbot là tin nhắn khách hàng: văn nói, teencode, viết tắt, không
dấu — gần với văn bản web hơn là tin tức, nên v2 có lợi thế hợp lý mà chi phí huấn luyện/suy luận không đổi. Muốn
so sánh, đổi `MODEL_NAME = "vinai/phobert-base"` và chạy lại (kết quả ghi rõ tên mô hình).

Tiêu chí chọn learning rate giống A1: **recall EMERGENCY trước, rồi macro-F1** trên val. Trong mỗi lần chạy,
early stopping theo **macro-F1 val** (giữ checkpoint tốt nhất).

In [ ]:
# ===================== Cấu hình =====================
MODEL_NAME = "vinai/phobert-base-v2"     # hoặc "vinai/phobert-base"
SEGMENTER = "vncorenlp"                  # "vncorenlp" (RDRSegmenter, như lúc tiền huấn luyện) | "underthesea" | "none"

DATA_SOURCE = "upload"                   # "upload" | "clone" | "drive"
REPO_URL = "https://github.com/<tai-khoan>/<ten-repo>.git"   # chỉ dùng khi DATA_SOURCE = "clone" (placeholder)
REPO_DATA_DIR = "ml/chat-intent/data"    # thư mục chứa train/val/test.csv trong repo
DRIVE_DATA_DIR = "/content/drive/MyDrive/vet-clinic-intent"  # chỉ dùng khi DATA_SOURCE = "drive"

MAX_LEN = 64
LEARNING_RATES = [2e-5, 3e-5, 5e-5]      # lưới nhỏ trong khoảng 2e-5..5e-5
NUM_EPOCHS = 10                          # tối đa; early stopping thường dừng sớm hơn
PATIENCE = 2                             # số epoch không cải thiện macro-F1 val thì dừng
BATCH_SIZE = 16
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
USE_CLASS_WEIGHTS = False                # True: cross-entropy có trọng số n/(k·n_lớp); dữ liệu khá cân bằng nên mặc định tắt
SEED = 42

CPU_THREADS = 2                          # mô phỏng VM nhỏ khi đo độ trễ CPU (Colab miễn phí cũng chỉ có 2 vCPU)
WORK_DIR = "/content/phobert-intent"
PIP_TRANSFORMERS = "transformers>=4.44,<5"   # ghim nhánh 4.x: API Trainer/eval_strategy và tokenizer PhoBERT (slow) ổn định

LABELS = sorted(["EMERGENCY", "OFF_TOPIC", "GREETING", "THANKS", "CONTACT", "OPENING_HOURS", "OTHER",
                 "PRICE", "SERVICE_INFO", "PRODUCT", "BOOKING", "CARE_ADVICE"])      # = rules.ALL_LABELS
RULE_LABELS = ["EMERGENCY", "OFF_TOPIC", "GREETING", "THANKS", "CONTACT", "OPENING_HOURS", "OTHER"]
ML_ONLY_LABELS = ["PRICE", "SERVICE_INFO", "PRODUCT", "BOOKING", "CARE_ADVICE"]

## 2. Cài thư viện

Colab đã có sẵn `torch`, `pandas`, `scikit-learn`. Ô này cài `transformers` 4.x, `accelerate` và bộ tách từ đã chọn.
`py_vncorenlp` chạy VnCoreNLP (Java) qua `pyjnius`, nên cần Java; ô sẽ cài OpenJDK nếu máy chưa có.

In [ ]:
import os, shutil, subprocess, sys

pkgs = [PIP_TRANSFORMERS, "accelerate>=0.26"]
if SEGMENTER == "vncorenlp":
    pkgs.append("py_vncorenlp")
elif SEGMENTER == "underthesea":
    pkgs.append("underthesea")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

if SEGMENTER == "vncorenlp" and shutil.which("java") is None:
    subprocess.check_call("apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless", shell=True)
if SEGMENTER == "vncorenlp" and not os.environ.get("JAVA_HOME") and shutil.which("java"):
    java = os.path.realpath(shutil.which("java"))
    os.environ["JAVA_HOME"] = os.path.dirname(os.path.dirname(java))

import torch, transformers, sklearn
print("torch", torch.__version__, "| transformers", transformers.__version__, "| sklearn", sklearn.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ (đổi runtime sang T4 GPU)")

## 3. Đưa dữ liệu vào Colab

Ba cách (chọn bằng `DATA_SOURCE`):

- `"upload"` (mặc định): chọn `train.csv`, `val.csv`, `test.csv` từ `ml/chat-intent/data/` trên máy.
- `"clone"`: clone repo. Repo riêng tư thì tạo GitHub token chỉ-đọc và lưu vào **Colab Secrets** với tên `GH_TOKEN`
  (biểu tượng chìa khoá bên trái) — không dán token vào notebook.
- `"drive"`: chép 3 file vào `DRIVE_DATA_DIR` trên Google Drive.

Cột: `text,label,source` (xem `LABELING.md`).

In [ ]:
import pandas as pd

DATA_DIR = os.path.join(WORK_DIR, "data")
RESULTS_DIR = os.path.join(WORK_DIR, "results")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
SPLITS = ["train", "val", "test"]

if DATA_SOURCE == "upload":
    from google.colab import files
    print("Chọn train.csv, val.csv, test.csv ...")
    uploaded = files.upload()
    for name, content in uploaded.items():
        base = os.path.basename(name)
        split = next((s for s in SPLITS if base.startswith(s)), None)   # Colab có thể đổi tên thành "train (1).csv"
        if split is None:
            print("bỏ qua", name)
            continue
        with open(os.path.join(DATA_DIR, f"{split}.csv"), "wb") as fh:
            fh.write(content)
elif DATA_SOURCE == "clone":
    url = REPO_URL
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
    except Exception:
        token = None
    if token and url.startswith("https://"):
        url = url.replace("https://", f"https://x-access-token:{token}@", 1)
    repo_dir = os.path.join(WORK_DIR, "repo")
    if not os.path.isdir(repo_dir):
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", url, repo_dir], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("git clone thất bại (kiểm tra REPO_URL / GH_TOKEN): "
                               + r.stderr.replace(token or "\0", "***"))
    for s in SPLITS:
        shutil.copy(os.path.join(repo_dir, REPO_DATA_DIR, f"{s}.csv"), os.path.join(DATA_DIR, f"{s}.csv"))
elif DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    for s in SPLITS:
        shutil.copy(os.path.join(DRIVE_DATA_DIR, f"{s}.csv"), os.path.join(DATA_DIR, f"{s}.csv"))
else:
    raise ValueError(DATA_SOURCE)


def read_split(name):
    df = pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv"), dtype=str, keep_default_na=False, encoding="utf-8")
    assert list(df.columns)[:3] == ["text", "label", "source"], df.columns
    bad = set(df["label"]) - set(LABELS)
    assert not bad, f"nhãn lạ trong {name}: {bad}"
    return df.reset_index(drop=True)


train_df, val_df, test_df = (read_split(s) for s in SPLITS)
for name, df in zip(SPLITS, (train_df, val_df, test_df)):
    print(f"{name}: {len(df)} câu |", dict(sorted(df["label"].value_counts().items())))

## 4. Tách từ (bắt buộc với PhoBERT)

PhoBERT được tiền huấn luyện trên văn bản **đã tách từ** bằng RDRSegmenter của VnCoreNLP: các âm tiết của một từ
ghép nối bằng `_` (`phòng khám` → `phòng_khám`, `đặt lịch` → `đặt_lịch`). Đưa câu chưa tách từ vào thì mô hình vẫn
chạy nhưng lệch phân phối so với lúc tiền huấn luyện, thường làm giảm điểm. Vì vậy:

- `vncorenlp` (mặc định): `py_vncorenlp` với annotator `wseg` — **đúng** bộ tách từ PhoBERT dùng. Nếu không khởi
  động được (thiếu Java, lỗi tải), ô tự chuyển sang `underthesea` và ghi rõ trong kết quả.
- `underthesea`: `word_tokenize(text, format="text")`, thuần Python, cũng nối bằng `_`, nhưng ranh giới từ có thể
  khác RDRSegmenter đôi chút.
- `none`: không tách từ — chỉ để thử nghiệm đối chứng.

**Điểm yếu với câu không dấu:** khoảng 30 % dữ liệu (27 % val) là câu không dấu / teencode (`dia chj phong kham o
duong nao`, `cn co lam ko shop`). Cả bộ tách từ lẫn từ vựng BPE của PhoBERT đều học trên văn bản **có dấu**, nên câu
không dấu gần như không được ghép từ, và mỗi âm tiết không dấu bị cắt thành các mảnh BPE ít nghĩa. Mô hình phải học
lại các mẫu này từ ~200 câu biến thể trong train. Notebook báo riêng chỉ số cho nhóm *có dấu* và *không dấu* để thấy
rõ điểm yếu này (A1 không bị ảnh hưởng vì `normalize()` bỏ dấu ở cả hai phía và dùng n-gram ký tự).
Không tự thêm dấu (khôi phục dấu) để giữ so sánh công bằng với A1/A2 trên cùng văn bản gốc.

In [ ]:
import time, unicodedata

VN_MARKS = set("àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ")


def has_diacritics(text):
    return any(c in VN_MARKS for c in unicodedata.normalize("NFC", text).lower())


SEGMENTER_USED = SEGMENTER
_segment_impl = None

if SEGMENTER == "vncorenlp":
    try:
        import py_vncorenlp
        VNC_DIR = "/content/vncorenlp"            # py_vncorenlp yêu cầu đường dẫn tuyệt đối
        os.makedirs(VNC_DIR, exist_ok=True)
        if not os.path.isdir(os.path.join(VNC_DIR, "models")):
            py_vncorenlp.download_model(save_dir=VNC_DIR)
        _cwd = os.getcwd()
        _rdr = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=VNC_DIR)
        os.chdir(_cwd)                            # py_vncorenlp tự đổi thư mục làm việc khi khởi tạo
        _segment_impl = lambda t: " ".join(_rdr.word_segment(t))
    except Exception as exc:                      # thiếu Java / lỗi tải → dùng underthesea
        print("Không khởi động được VnCoreNLP:", repr(exc), "→ chuyển sang underthesea")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "underthesea"])
        SEGMENTER_USED = "underthesea (dự phòng)"

if _segment_impl is None and SEGMENTER_USED.startswith("underthesea"):
    from underthesea import word_tokenize
    _segment_impl = lambda t: word_tokenize(t, format="text")
if _segment_impl is None:
    SEGMENTER_USED = "none"
    _segment_impl = lambda t: t


def segment(text):
    text = unicodedata.normalize("NFC", text).strip()
    if not text:
        return ""
    try:
        return _segment_impl(text)
    except Exception:
        return text                               # câu chỉ có emoji/ký hiệu lạ: giữ nguyên


t0 = time.perf_counter()
for df in (train_df, val_df, test_df):
    df["seg"] = [segment(t) for t in df["text"]]
    df["accented"] = [has_diacritics(t) for t in df["text"]]
print(f"Bộ tách từ: {SEGMENTER_USED} | {time.perf_counter() - t0:.1f} s cho {len(train_df) + len(val_df) + len(test_df)} câu")
print("Tỷ lệ câu không dấu: train %.2f, val %.2f, test %.2f" %
      tuple(1 - df["accented"].mean() for df in (train_df, val_df, test_df)))
for t in ["Em muốn đặt lịch siêu âm cho mèo đang mang thai", "dia chj phong kham o duong nao", "cn co lam ko shop"]:
    print(f"  {t!r:55} → {segment(t)!r}")

## 5. Tokenizer, dữ liệu, chỉ số

Chỉ số giống `common.summarize()` của A1: accuracy, macro-F1 (trên các nhãn có mặt), precision/recall/F1 từng nhãn,
recall/precision/báo nhầm của EMERGENCY, ma trận nhầm lẫn; thêm bản 7 nhãn (gộp 5 nhãn mới → OTHER) như `eval.py`.

In [ ]:
import numpy as np
from collections import Counter
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)

set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)     # PhoBERT chỉ có tokenizer "slow" (BPE trên văn bản đã tách từ)
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}


class IntentDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tokenizer(df["seg"].tolist(), truncation=True, max_length=MAX_LEN)
        self.y = [label2id[l] for l in df["label"]]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i]
        return item


train_ds, val_ds, test_ds = IntentDataset(train_df), IntentDataset(val_df), IntentDataset(test_df)
collator = DataCollatorWithPadding(tokenizer)

lens = [len(ids) for ids in train_ds.enc["input_ids"]]
unk = tokenizer.unk_token_id
unk_rate = sum(ids.count(unk) for ids in train_ds.enc["input_ids"]) / max(1, sum(lens))
print(f"Độ dài token (train): trung vị {int(np.median(lens))}, max {max(lens)}, "
      f"bị cắt ở {MAX_LEN}: {sum(l >= MAX_LEN for l in lens)} câu, tỷ lệ <unk>: {unk_rate:.3%}")
for flag, name in ((True, "có dấu"), (False, "không dấu")):
    sub = [l for l, a in zip(lens, train_df["accented"]) if a == flag]
    words = [len(s.split()) for s, a in zip(train_df["text"], train_df["accented"]) if a == flag]
    print(f"  {name}: {np.mean(sub):.1f} token/câu, {np.mean(sub) / max(1e-9, np.mean(words)):.2f} token/âm tiết")


def to_rule_space(label):
    return "OTHER" if label in ML_ONLY_LABELS else label


def summarize(y_true, y_pred, labels):
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    per = {lab: {"precision": p[i], "recall": r[i], "f1": f[i], "support": int(s[i])} for i, lab in enumerate(labels)}
    present = [lab for lab in labels if per[lab]["support"] > 0]
    out = {"n": len(y_true), "accuracy": accuracy_score(y_true, y_pred) if len(y_true) else 0.0,
           "macro_f1": f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0) if present else 0.0,
           "per_label": per, "labels": labels,
           "confusion": confusion_matrix(y_true, y_pred, labels=labels) if len(y_true) else None}
    if "EMERGENCY" in labels:
        tp = sum(1 for a, b in zip(y_true, y_pred) if a == "EMERGENCY" and b == "EMERGENCY")
        fn = sum(1 for a, b in zip(y_true, y_pred) if a == "EMERGENCY" and b != "EMERGENCY")
        fp = sum(1 for a, b in zip(y_true, y_pred) if a != "EMERGENCY" and b == "EMERGENCY")
        out["emergency"] = {"recall": tp / (tp + fn) if tp + fn else 0.0, "precision": tp / (tp + fp) if tp + fp else 0.0,
                            "tp": tp, "fn": fn, "false_alarms": fp}
    return out


def format_report(m, title):
    lines = [f"### {title}", "", f"n = {m['n']}  |  accuracy = {m['accuracy']:.4f}  |  macro-F1 = {m['macro_f1']:.4f}", ""]
    if "emergency" in m:
        e = m["emergency"]
        lines += [f"EMERGENCY: recall = {e['recall']:.4f} ({e['tp']}/{e['tp'] + e['fn']}), "
                  f"precision = {e['precision']:.4f}, báo nhầm (FP) = {e['false_alarms']}", ""]
    lines += ["| nhãn | precision | recall | F1 | support |", "|---|---:|---:|---:|---:|"]
    for lab in m["labels"]:
        d = m["per_label"][lab]
        lines.append(f"| {lab} | {d['precision']:.3f} | {d['recall']:.3f} | {d['f1']:.3f} | {d['support']} |")
    if m["confusion"] is not None:
        short = [lab[:6] for lab in m["labels"]]
        lines += ["", "Ma trận nhầm lẫn (hàng = thật, cột = dự đoán):", "", "```",
                  "thật\\dự".ljust(14) + "".join(s.rjust(7) for s in short)]
        for lab, row in zip(m["labels"], m["confusion"]):
            lines.append(lab[:13].ljust(14) + "".join(str(v).rjust(7) for v in row))
        lines.append("```")
    return "\n".join(lines) + "\n"


EM_ID = label2id["EMERGENCY"]


def compute_metrics(eval_pred):
    logits, y = eval_pred
    logits = logits[0] if isinstance(logits, tuple) else logits
    pred = logits.argmax(-1)
    present = sorted(set(y.tolist()))
    tp = int(((y == EM_ID) & (pred == EM_ID)).sum())
    return {"accuracy": float((pred == y).mean()),
            "macro_f1": float(f1_score(y, pred, labels=present, average="macro", zero_division=0)),
            "em_recall": tp / max(1, int((y == EM_ID).sum())),
            "em_false_alarms": int(((y != EM_ID) & (pred == EM_ID)).sum())}

## 6. Huấn luyện (lưới learning rate + early stopping)

Mỗi learning rate: tối đa `NUM_EPOCHS` epoch, đánh giá val sau mỗi epoch, dừng khi macro-F1 val không tăng sau
`PATIENCE` epoch, nạp lại checkpoint tốt nhất. Giữa các learning rate chọn theo (recall EMERGENCY, macro-F1) trên val.
Seed cố định (`SEED`) cho khởi tạo đầu phân loại, thứ tự batch và dropout; fp16 trên GPU vẫn có thể lệch rất nhỏ
giữa các lần chạy.

In [ ]:
class WeightedTrainer(Trainer):
    """Trainer với cross-entropy có trọng số lớp (chỉ dùng khi USE_CLASS_WEIGHTS = True)."""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        loss = torch.nn.functional.cross_entropy(outputs.logits.float(), labels, weight=w)
        return (loss, outputs) if return_outputs else loss


class_weights = None
if USE_CLASS_WEIGHTS:
    cnt = Counter(train_df["label"])
    class_weights = torch.tensor([len(train_df) / (len(LABELS) * cnt[l]) for l in LABELS], dtype=torch.float)

BEST_DIR = os.path.join(WORK_DIR, "best-model")
grid, best = [], None
for lr in LEARNING_RATES:
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
    run_dir = os.path.join(WORK_DIR, "runs", f"lr{lr:g}")
    args = TrainingArguments(
        output_dir=run_dir, eval_strategy="epoch", save_strategy="epoch", logging_strategy="epoch",
        learning_rate=lr, num_train_epochs=NUM_EPOCHS, per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=64, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        save_total_limit=2, fp16=torch.cuda.is_available(), seed=SEED, data_seed=SEED, report_to="none")
    trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                              data_collator=collator, compute_metrics=compute_metrics,
                              callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
                              class_weights=class_weights)
    t0 = time.perf_counter()
    trainer.train()
    secs = time.perf_counter() - t0
    ev = trainer.evaluate()                       # trên checkpoint tốt nhất đã nạp lại
    epochs_run = max((h.get("epoch", 0) for h in trainer.state.log_history), default=0)
    best_epoch = next((h["epoch"] for h in trainer.state.log_history
                       if "eval_macro_f1" in h and trainer.state.best_metric is not None
                       and abs(h["eval_macro_f1"] - trainer.state.best_metric) < 1e-9), None)
    row = {"lr": lr, "best_epoch": best_epoch, "epochs_run": epochs_run, "acc": ev["eval_accuracy"],
           "macro_f1": ev["eval_macro_f1"], "em_recall": ev["eval_em_recall"],
           "em_fp": ev["eval_em_false_alarms"], "secs": secs}
    grid.append(row)
    print(row)
    if best is None or (row["em_recall"], row["macro_f1"]) > (best["em_recall"], best["macro_f1"]):
        best = row
        trainer.save_model(BEST_DIR)
        tokenizer.save_pretrained(BEST_DIR)
    del trainer, model
    shutil.rmtree(run_dir, ignore_errors=True)    # mỗi checkpoint ~1,5 GB (kèm optimizer)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Chọn:", best)

## 7. Đánh giá trên val

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model_mb = sum(os.path.getsize(os.path.join(BEST_DIR, f)) for f in os.listdir(BEST_DIR)) / 2**20
n_params = sum(p.numel() for p in model.parameters())
print(f"Mô hình: {n_params / 1e6:.0f} triệu tham số, thư mục lưu {model_mb:.0f} MB")


@torch.no_grad()
def predict(seg_texts, device=DEVICE, bs=64):
    model.to(device).eval()
    out = []
    for i in range(0, len(seg_texts), bs):
        enc = tokenizer(seg_texts[i:i + bs], truncation=True, max_length=MAX_LEN, padding=True,
                        return_tensors="pt").to(device)
        out.append(torch.softmax(model(**enc).logits.float(), -1).cpu().numpy())
    probs = np.concatenate(out)
    return [LABELS[k] for k in probs.argmax(1)], probs.max(1)


val_pred, val_conf = predict(val_df["seg"].tolist())
val_m = summarize(val_df["label"].tolist(), val_pred, LABELS)
print(format_report(val_m, "PhoBERT — val"))

## 8. Độ trễ suy luận (từng câu, batch = 1)

Đo trên 154 câu val, mỗi câu riêng lẻ như khi chatbot nhận một tin nhắn: **tách từ** + **token hoá & mô hình**.
GPU: fp32, có `torch.cuda.synchronize()`. CPU: fp32, `torch.set_num_threads(CPU_THREADS)` để gần với VM nhỏ của hệ
thống (A1 trong `results/a1.md` đo bằng Python thuần trên máy dev: ~0,2 ms/câu).

In [ ]:
def bench(device, n_warmup=10):
    texts = val_df["text"].tolist()
    model.to(device).eval()
    seg_ms, mdl_ms = [], []
    with torch.no_grad():
        for i, t in enumerate(texts[:n_warmup] + texts):
            t0 = time.perf_counter()
            s = segment(t)
            t1 = time.perf_counter()
            enc = tokenizer([s], truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
            model(**enc)
            if device == "cuda":
                torch.cuda.synchronize()
            t2 = time.perf_counter()
            if i >= n_warmup:
                seg_ms.append((t1 - t0) * 1000)
                mdl_ms.append((t2 - t1) * 1000)
    tot = np.array(seg_ms) + np.array(mdl_ms)
    return {"seg_med": float(np.median(seg_ms)), "mdl_med": float(np.median(mdl_ms)),
            "mdl_p95": float(np.percentile(mdl_ms, 95)), "tot_med": float(np.median(tot)),
            "tot_p95": float(np.percentile(tot, 95)), "n": len(tot)}


latency = {}
if torch.cuda.is_available():
    latency["GPU " + torch.cuda.get_device_name(0)] = bench("cuda")
_threads = torch.get_num_threads()
torch.set_num_threads(CPU_THREADS)
latency[f"CPU ({CPU_THREADS} luồng)"] = bench("cpu")
torch.set_num_threads(_threads)
CPU_KEY = f"CPU ({CPU_THREADS} luồng)"
model.to(DEVICE)
for k, v in latency.items():
    print(k, {a: round(b, 2) for a, b in v.items()})

## 9. Đánh giá trên test — **chỉ chạy MỘT lần**

Chỉ chạy sau khi đã chốt cấu hình ở các bước trên. Ô ghi dấu `TEST_DA_CHAY` vào `WORK_DIR`; chạy lại sẽ báo lỗi
(trừ khi đặt `FORCE_RERUN_TEST = True`, và khi đó phải ghi rõ trong báo cáo). Không quay lại chỉnh siêu tham số
theo kết quả test.

In [ ]:
FORCE_RERUN_TEST = False
TEST_MARKER = os.path.join(WORK_DIR, "TEST_DA_CHAY")
if os.path.exists(TEST_MARKER) and not FORCE_RERUN_TEST:
    raise RuntimeError("Tập test đã được đánh giá một lần (" + open(TEST_MARKER).read().strip() + ").")
test_pred, test_conf = predict(test_df["seg"].tolist())
test_m = summarize(test_df["label"].tolist(), test_pred, LABELS)
with open(TEST_MARKER, "w") as fh:
    fh.write(time.strftime("%Y-%m-%d %H:%M:%S"))
print(format_report(test_m, "PhoBERT — test"))

## 10. Ghi `results/phobert.md` và tải về

Bảng chính có cùng cột với `results/a1.md`; cột `ms/câu` là **trung vị CPU** (tách từ + mô hình) vì đó là môi trường
triển khai. Tải file về và chép vào `ml/chat-intent/results/phobert.md`.

In [ ]:
def split_section(name, df, pred, conf):
    gold = df["label"].tolist()
    m12 = summarize(gold, pred, LABELS)
    m7 = summarize([to_rule_space(g) for g in gold], [to_rule_space(p) for p in pred], RULE_LABELS)
    e = m12["emergency"]
    ms = latency[CPU_KEY]["tot_med"]
    lines = [f"## {name} (n = {len(df)})", "",
             "| hệ | acc (12) | macro-F1 (12) | acc (7) | macro-F1 (7) | EMERG precision | EMERG recall | EMERG báo nhầm | ms/câu |",
             "|---|---:|---:|---:|---:|---:|---:|---:|---:|",
             f"| phobert | {m12['accuracy']:.4f} | {m12['macro_f1']:.4f} | {m7['accuracy']:.4f} | {m7['macro_f1']:.4f} | "
             f"{e['precision']:.4f} | {e['recall']:.4f} ({e['tp']}/{e['tp'] + e['fn']}) | {e['false_alarms']} | {ms:.1f} |",
             "", "(7) = gộp PRICE/SERVICE_INFO/PRODUCT/BOOKING/CARE_ADVICE → OTHER ở cả nhãn thật lẫn dự đoán.",
             f"ms/câu = trung vị CPU {CPU_THREADS} luồng, gồm tách từ + mô hình (xem mục Độ trễ).", "",
             "Theo nhóm câu:", "", "| nhóm | n | acc | macro-F1 | EMERG recall |", "|---|---:|---:|---:|---:|"]
    groups = [(f"source = {s}", (df["source"] == s).values) for s in sorted(df["source"].unique())]
    groups += [("có dấu", df["accented"].values), ("không dấu", ~df["accented"].values)]
    for gname, mask in groups:
        g = [x for x, k in zip(gold, mask) if k]
        p = [x for x, k in zip(pred, mask) if k]
        if not g:
            continue
        mg = summarize(g, p, LABELS)
        em = mg["emergency"]
        emr = f"{em['recall']:.4f} ({em['tp']}/{em['tp'] + em['fn']})" if em["tp"] + em["fn"] else "–"
        lines.append(f"| {gname} | {len(g)} | {mg['accuracy']:.4f} | {mg['macro_f1']:.4f} | {emr} |")
    wrong = [(t, g, p, c) for t, g, p, c in zip(df["text"], gold, pred, conf) if g != p]
    lines += ["", f"PhoBERT sai {len(wrong)}/{len(df)} câu (tối đa 20):", "",
              "| câu | thật | PhoBERT (conf) |", "|---|---|---|"]
    for t, g, p, c in wrong[:20]:
        lines.append(f"| {t.replace('|', '/')} | {g} | {p} ({c:.2f}) |")
    return "\n".join(lines) + "\n\n" + format_report(m12, f"PhoBERT chi tiết — {name}")


parts = [
    "# A3 — PhoBERT fine-tune (chỉ để so sánh, không triển khai)", "",
    f"Mô hình: `{MODEL_NAME}` ({n_params / 1e6:.0f} triệu tham số, {model_mb:.0f} MB), tách từ `{SEGMENTER_USED}`, "
    f"max_len {MAX_LEN}, batch {BATCH_SIZE}, weight decay {WEIGHT_DECAY}, warmup {WARMUP_RATIO}, "
    f"class weights = {USE_CLASS_WEIGHTS}, seed {SEED}.",
    f"Môi trường: Google Colab, {next((k for k in latency if k.startswith('GPU')), 'không GPU')}, "
    f"torch {torch.__version__}, transformers {transformers.__version__}; chạy lúc {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}.",
    "",
    f"Chọn trên val: lr = {best['lr']:g}, epoch tốt nhất = {best['best_epoch']} (macro-F1 val {best['macro_f1']:.4f}, "
    f"recall EMERGENCY {best['em_recall']:.4f}). Early stopping theo macro-F1 val, patience {PATIENCE}, tối đa {NUM_EPOCHS} epoch.",
    "Siêu tham số chọn trên val rồi báo cáo cũng trên val ⇒ số val hơi lạc quan (giống A1/A2); test chỉ chạy một lần.",
    "", "Lưới learning rate (val): lr | epoch tốt nhất | số epoch đã chạy | acc | macro-F1 | EM recall | EM FP | giây huấn luyện",
    "", "```"]
for r in grid:
    parts.append(f"{r['lr']:.0e} | {r['best_epoch']} | {r['epochs_run']:.0f} | {r['acc']:.4f} | {r['macro_f1']:.4f} | "
                 f"{r['em_recall']:.4f} | {r['em_fp']} | {r['secs']:.0f}")
parts += ["```", "", split_section("val", val_df, val_pred, val_conf)]
if "test_pred" in globals():
    parts.append(split_section("test", test_df, test_pred, test_conf))
else:
    parts.append("## test\n\nChưa chạy.\n")
parts += ["## Độ trễ suy luận (từng câu, batch = 1, 154 câu val)", "",
          "| thiết bị | tách từ (trung vị) | token hoá + mô hình (trung vị) | mô hình p95 | tổng (trung vị) | tổng p95 |",
          "|---|---:|---:|---:|---:|---:|"]
for k, v in latency.items():
    parts.append(f"| {k} | {v['seg_med']:.2f} ms | {v['mdl_med']:.2f} ms | {v['mdl_p95']:.2f} ms | "
                 f"{v['tot_med']:.2f} ms | {v['tot_p95']:.2f} ms |")
parts += ["", "So sánh: A1 ~0,2 ms/câu (Python thuần, JSON 1 MB, không cần thư viện ML ở Java). "
          "PhoBERT cần PyTorch (hoặc ONNX Runtime) và ~540 MB trọng số, nên chỉ dùng để so sánh trong báo cáo.", ""]

md_text = "\n".join(parts)
out_md = os.path.join(RESULTS_DIR, "phobert.md")
with open(out_md, "w", encoding="utf-8") as fh:
    fh.write(md_text)
pred_rows = [pd.DataFrame({"split": "val", "text": val_df["text"], "label": val_df["label"], "pred": val_pred, "conf": val_conf})]
if "test_pred" in globals():
    pred_rows.append(pd.DataFrame({"split": "test", "text": test_df["text"], "label": test_df["label"],
                                   "pred": test_pred, "conf": test_conf}))
pd.concat(pred_rows).to_csv(os.path.join(RESULTS_DIR, "phobert-predictions.csv"), index=False, encoding="utf-8")
print(md_text)

In [ ]:
# Tải kết quả về máy: chép phobert.md vào ml/chat-intent/results/
try:
    from google.colab import files
    files.download(os.path.join(RESULTS_DIR, "phobert.md"))
    files.download(os.path.join(RESULTS_DIR, "phobert-predictions.csv"))
except ImportError:
    print("Không chạy trên Colab; kết quả ở", RESULTS_DIR)

## 11. (Tuỳ chọn) Lưu mô hình lên Google Drive

Mô hình **không** dùng trong hệ thống; chỉ lưu nếu muốn chạy lại đánh giá mà không huấn luyện lại. Không commit
trọng số vào repo.

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = "/content/drive/MyDrive/vet-clinic-intent/phobert-best"
    shutil.copytree(BEST_DIR, dst, dirs_exist_ok=True)
    print("Đã lưu", dst)